# Trích xuất dữ liệu thời tiết - 13 tỉnh thành ĐBSCL

Trích xuất và hợp nhất dữ liệu từ các file CSV/XLSX cho **13 tỉnh thành Đồng bằng sông Cửu Long**:

> Long An, Tiền Giang, Bến Tre, Đồng Tháp, Vĩnh Long, Trà Vinh, Hậu Giang, Sóc Trăng, Bạc Liêu, An Giang, Kiên Giang, Cà Mau, Cần Thơ

Đầu ra: mỗi tỉnh xuất **1 file CSV** có tên định dạng `{tinh}_{YYYYMMDD}.csv`


In [1]:
import os, re, glob
from datetime import datetime
import pandas as pd
import numpy as np

BASE_DIR   = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
CRAWL_DIR  = os.path.join(BASE_DIR, "data", "data_crawl")
CLEAN_DIR  = os.path.join(BASE_DIR, "data", "data_clean", "data_merge_clean")
OUTPUT_DIR = os.path.join(BASE_DIR, "data", "data_dbscl")
os.makedirs(OUTPUT_DIR, exist_ok=True)

DBSCL_PROVINCES = [
    "Long An", "Tien Giang", "Ben Tre", "Dong Thap", "Vinh Long",
    "Tra Vinh", "Hau Giang", "Soc Trang", "Bac Lieu",
    "An Giang", "Kien Giang", "Ca Mau", "Can Tho",
]

DBSCL_ALIASES = {
    "long an":     "Long An",
    "tien giang":  "Tien Giang",  "tien giang":  "Tien Giang",
    "ben tre":     "Ben Tre",
    "dong thap":   "Dong Thap",
    "vinh long":   "Vinh Long",
    "tra vinh":    "Tra Vinh",
    "hau giang":   "Hau Giang",
    "soc trang":   "Soc Trang",
    "bac lieu":    "Bac Lieu",
    "an giang":    "An Giang",
    "kien giang":  "Kien Giang",
    "ca mau":      "Ca Mau",
    "can tho":     "Can Tho",   "thanh pho can tho": "Can Tho",
}

def norm(s):
    if pd.isna(s): return ""
    t = str(s).strip().lower()
    t = re.sub(r"\s+", " ", t)
    return t

TODAY_STR = datetime.now().strftime("%Y%m%d")
print("BASE_DIR   :", BASE_DIR)
print("CRAWL_DIR  :", CRAWL_DIR)
print("CLEAN_DIR  :", CLEAN_DIR)
print("OUTPUT_DIR :", OUTPUT_DIR)
print(f"Run date   : {TODAY_STR}  |  provinces: {len(DBSCL_PROVINCES)}")


BASE_DIR   : d:\Hoc_tap\PROJECT_WEATHER_FORCAST
CRAWL_DIR  : d:\Hoc_tap\PROJECT_WEATHER_FORCAST\data\data_crawl
CLEAN_DIR  : d:\Hoc_tap\PROJECT_WEATHER_FORCAST\data\data_clean\data_merge_clean
OUTPUT_DIR : d:\Hoc_tap\PROJECT_WEATHER_FORCAST\data\data_dbscl
Run date   : 20260319  |  provinces: 13


In [2]:

# --- Cell 2: Ham doc file & quet thu muc ---

FULL_COLS = [
    "station_id","station_name","province","district",
    "latitude","longitude","timestamp","data_source","data_quality",
    "data_time","temperature_current","temperature_max","temperature_min","temperature_avg",
    "humidity_current","humidity_max","humidity_min","humidity_avg",
    "pressure_current","pressure_max","pressure_min","pressure_avg",
    "wind_speed_current","wind_speed_max","wind_speed_min","wind_speed_avg",
    "wind_direction_current","wind_direction_avg",
    "rain_current","rain_max","rain_min","rain_avg","rain_total",
    "cloud_cover_current","cloud_cover_max","cloud_cover_min","cloud_cover_avg",
    "visibility_current","visibility_max","visibility_min","visibility_avg",
    "thunder_probability","status",
]

RAIN_COLS = [
    "station_id","station_name","province","district",
    "rain_total","status","timestamp","data_time",
]


def read_one_file(path):
    """
    Doc 1 file CSV hoac XLSX.
    Tra ve DataFrame hoac None neu loi.
    """
    try:
        ext = os.path.splitext(path)[1].lower()
        if ext == ".xlsx" or ext == ".xls":
            df = pd.read_excel(path, dtype=str)
        elif ext == ".csv":
            for enc in ("utf-8-sig", "utf-8", "cp1258", "latin-1"):
                try:
                    df = pd.read_csv(path, dtype=str, encoding=enc)
                    break
                except UnicodeDecodeError:
                    continue
            else:
                print(f"  [WARN] Khong doc duoc encoding: {path}")
                return None
        else:
            return None

        if df is None or df.empty or "province" not in df.columns:
            return None

        df["_source_file"] = os.path.basename(path)
        return df
    except Exception as e:
        print(f"  [ERR] {os.path.basename(path)}: {e}")
        return None


def extract_date_from_filename(fname):
    """Lay YYYYMMDD tu Bao_cao_YYYYMMDD_HHMMSS(.csv|.xlsx)"""
    m = re.search(r"(\d{8})_\d{6}", fname)
    return m.group(1) if m else None


def scan_directory(directory, extensions=(".csv", ".xlsx", ".xls")):
    """Lay tat ca file co extension cho truoc trong thu muc."""
    files = []
    for ext in extensions:
        files.extend(glob.glob(os.path.join(directory, f"*{ext}")))
    return sorted(files)


crawl_files = scan_directory(CRAWL_DIR)
clean_files = scan_directory(CLEAN_DIR)
print(f"data_crawl : {len(crawl_files)} file")
print(f"data_clean : {len(clean_files)} file")


data_crawl : 289 file
data_clean : 2 file


In [3]:

# --- Cell 3: Doc toan bo & loc theo 13 tinh DBSCL ---

import unicodedata

def remove_accents(s):
    """Chuyen chu co dau -> khong dau de so sanh."""
    if not isinstance(s, str):
        return ""
    nfkd = unicodedata.normalize("NFKD", s)
    return "".join(c for c in nfkd if not unicodedata.combining(c)).lower().strip()

# Tap hop chu khong dau cua 13 tinh de loc
DBSCL_NO_ACCENT = {
    remove_accents(p): p
    for p in [
        "Long An", "Tien Giang", "Ben Tre", "Dong Thap", "Vinh Long",
        "Tra Vinh", "Hau Giang", "Soc Trang", "Bac Lieu",
        "An Giang", "Kien Giang", "Ca Mau", "Can Tho",
    ]
}
# Bo sung bien the "Thanh pho Can Tho"
DBSCL_NO_ACCENT["thanh pho can tho"] = "Can Tho"
DBSCL_NO_ACCENT["tp. can tho"]       = "Can Tho"
DBSCL_NO_ACCENT["tp can tho"]        = "Can Tho"

def match_province(raw_province):
    """Tra ve ten chuan hoac None neu khong thuoc DBSCL."""
    key = remove_accents(str(raw_province) if not pd.isna(raw_province) else "")
    # khop chinh xac
    if key in DBSCL_NO_ACCENT:
        return DBSCL_NO_ACCENT[key]
    # khop chua (xu ly "Tinh Long An" -> "Long An")
    for k, v in DBSCL_NO_ACCENT.items():
        if k and k in key:
            return v
    return None


all_frames = []
skipped = 0
all_files = crawl_files + clean_files

print(f"Bat dau doc {len(all_files)} file ...")
for fpath in all_files:
    df = read_one_file(fpath)
    if df is None:
        skipped += 1
        continue

    df["_province_std"] = df["province"].apply(match_province)
    df_filt = df[df["_province_std"].notna()].copy()

    if df_filt.empty:
        skipped += 1
        continue

    fname = os.path.basename(fpath)
    date_str = extract_date_from_filename(fname)
    df_filt["_file_date"] = date_str if date_str else pd.NaT
    all_frames.append(df_filt)

if all_frames:
    combined = pd.concat(all_frames, ignore_index=True, sort=False)
    print(f"Tong so dong DBSCL (truoc dedup): {len(combined):,}")
    print("Phan bo theo tinh:")
    print(combined["_province_std"].value_counts().to_string())
else:
    combined = pd.DataFrame()
    print("CANH BAO: Khong tim thay du lieu DBSCL nao!")
print(f"File bo qua: {skipped}")


Bat dau doc 291 file ...
  [ERR] ~$Bao_cao_20260306_155820.xlsx: Excel file format cannot be determined, you must specify an engine manually.
  [ERR] ~$Bao_cao_20260306_160220 (Copy).xlsx: Excel file format cannot be determined, you must specify an engine manually.
  [ERR] ~$Bao_cao_20260306_160220.xlsx: Excel file format cannot be determined, you must specify an engine manually.
  [ERR] ~$Bao_cao_20260308_125220.xlsx: Excel file format cannot be determined, you must specify an engine manually.
  [ERR] ~$Bao_cao_20260310_145452.xlsx: Excel file format cannot be determined, you must specify an engine manually.
Tong so dong DBSCL (truoc dedup): 73,728
Phan bo theo tinh:
_province_std
Ca Mau        14594
Can Tho       13194
An Giang       9182
Hau Giang      6039
Bac Lieu       5635
Kien Giang     5476
Vinh Long      4990
Tien Giang     4936
Soc Trang      3757
Ben Tre        3555
Long An        1185
Tra Vinh       1185
File bo qua: 7


In [4]:

# --- Cell 4: Xu ly, hop nhat cot & khu trung ---

def safe_float(s):
    try:
        return float(s)
    except Exception:
        return np.nan

# Cot so hoc -- ep kieu float
NUMERIC_COLS = [
    "latitude","longitude",
    "temperature_current","temperature_max","temperature_min","temperature_avg",
    "humidity_current","humidity_max","humidity_min","humidity_avg",
    "pressure_current","pressure_max","pressure_min","pressure_avg",
    "wind_speed_current","wind_speed_max","wind_speed_min","wind_speed_avg",
    "wind_direction_current","wind_direction_avg",
    "rain_current","rain_max","rain_min","rain_avg","rain_total",
    "cloud_cover_current","cloud_cover_max","cloud_cover_min","cloud_cover_avg",
    "visibility_current","visibility_max","visibility_min","visibility_avg",
    "thunder_probability",
]

if not combined.empty:
    for col in NUMERIC_COLS:
        if col in combined.columns:
            combined[col] = pd.to_numeric(combined[col], errors="coerce")

    # Chuan hoa timestamp -> datetime
    for dt_col in ("timestamp", "data_time"):
        if dt_col in combined.columns:
            combined[dt_col] = pd.to_datetime(combined[dt_col], errors="coerce", utc=False)

    # Khu trung: giu ban ghi moi nhat theo (station_id, data_time)
    dedup_keys = ["station_id", "data_time"]
    valid_dedup = [k for k in dedup_keys if k in combined.columns]
    if valid_dedup:
        before = len(combined)
        combined.sort_values("timestamp", na_position="last", inplace=True)
        combined.drop_duplicates(subset=valid_dedup, keep="last", inplace=True)
        print(f"Khu trung: {before:,} -> {len(combined):,} dong (bo {before - len(combined):,})")

    # Gan lai province = ten chuan (bo dau)
    combined["province_std"] = combined["_province_std"]

    print(f"\nTong sau khu trung: {len(combined):,} dong")
    print("Phan bo cuoi:")
    print(combined["province_std"].value_counts().to_string())
else:
    print("Khong co du lieu de xu ly.")


Khu trung: 73,728 -> 5,433 dong (bo 68,295)

Tong sau khu trung: 5,433 dong
Phan bo cuoi:
province_std
Kien Giang    797
An Giang      606
Ca Mau        588
Can Tho       572
Soc Trang     529
Tien Giang    512
Ben Tre       429
Hau Giang     377
Vinh Long     369
Bac Lieu      368
Long An       145
Tra Vinh      141


In [5]:

# --- Cell 5: Xuat CSV theo tung tinh co bang ngay thang ---

EXPORT_COLS_PRIORITY = [
    "station_id","station_name","province","province_std","district",
    "latitude","longitude","timestamp","data_time","data_source","data_quality",
    "temperature_current","temperature_max","temperature_min","temperature_avg",
    "humidity_current","humidity_max","humidity_min","humidity_avg",
    "pressure_current","pressure_max","pressure_min","pressure_avg",
    "wind_speed_current","wind_speed_max","wind_speed_min","wind_speed_avg",
    "wind_direction_current","wind_direction_avg",
    "rain_current","rain_max","rain_min","rain_avg","rain_total",
    "cloud_cover_current","cloud_cover_max","cloud_cover_min","cloud_cover_avg",
    "visibility_current","visibility_max","visibility_min","visibility_avg",
    "thunder_probability","status",
    "_source_file","_file_date",
]

exported_files = []

if not combined.empty:
    for prov in sorted(combined["province_std"].dropna().unique()):
        df_prov = combined[combined["province_std"] == prov].copy()

        # Lay cot co trong du lieu (theo thu tu uu tien)
        cols_out = [c for c in EXPORT_COLS_PRIORITY if c in df_prov.columns]
        extra    = [c for c in df_prov.columns if c not in cols_out]
        df_prov  = df_prov[cols_out + extra]

        # Sort theo thoi gian
        if "timestamp" in df_prov.columns:
            df_prov.sort_values("timestamp", inplace=True)

        # Ten file: {tinh_no_space}_{YYYYMMDD}.csv
        safe_name = prov.replace(" ", "_")
        out_name  = f"{safe_name}_{TODAY_STR}.csv"
        out_path  = os.path.join(OUTPUT_DIR, out_name)

        df_prov.to_csv(out_path, index=False, encoding="utf-8-sig")
        exported_files.append((prov, len(df_prov), out_path))
        print(f"  [{prov:15s}] {len(df_prov):5,} dong  ->  {out_name}")

    print(f"\nDa xuat {len(exported_files)} file vao: {OUTPUT_DIR}")
else:
    print("Khong co du lieu de xuat.")


  [An Giang       ]   606 dong  ->  An_Giang_20260319.csv
  [Bac Lieu       ]   368 dong  ->  Bac_Lieu_20260319.csv
  [Ben Tre        ]   429 dong  ->  Ben_Tre_20260319.csv
  [Ca Mau         ]   588 dong  ->  Ca_Mau_20260319.csv
  [Can Tho        ]   572 dong  ->  Can_Tho_20260319.csv
  [Hau Giang      ]   377 dong  ->  Hau_Giang_20260319.csv
  [Kien Giang     ]   797 dong  ->  Kien_Giang_20260319.csv
  [Long An        ]   145 dong  ->  Long_An_20260319.csv
  [Soc Trang      ]   529 dong  ->  Soc_Trang_20260319.csv
  [Tien Giang     ]   512 dong  ->  Tien_Giang_20260319.csv
  [Tra Vinh       ]   141 dong  ->  Tra_Vinh_20260319.csv
  [Vinh Long      ]   369 dong  ->  Vinh_Long_20260319.csv

Da xuat 12 file vao: d:\Hoc_tap\PROJECT_WEATHER_FORCAST\data\data_dbscl


In [6]:

# --- Cell 6: Kiem tra ket qua ---

if exported_files:
    summary = pd.DataFrame(exported_files, columns=["Province", "Records", "File"])
    summary["File"] = summary["File"].apply(os.path.basename)
    print("=== Ket qua xuat file ===")
    print(summary.to_string(index=False))
    print(f"\nTong records DBSCL: {summary['Records'].sum():,}")
    print(f"Thu muc output    : {OUTPUT_DIR}")

    # Danh sach tinh chua co du lieu
    prov_exported = set(summary["Province"].tolist())
    prov_all = {
        "Long An", "Tien Giang", "Ben Tre", "Dong Thap", "Vinh Long",
        "Tra Vinh", "Hau Giang", "Soc Trang", "Bac Lieu",
        "An Giang", "Kien Giang", "Ca Mau", "Can Tho",
    }
    missing = prov_all - prov_exported
    if missing:
        print(f"\nCac tinh CHUA CO du lieu ({len(missing)}):")
        for p in sorted(missing):
            print(f"  - {p}")
    else:
        print("\nTat ca 13 tinh deu co du lieu!")
else:
    print("Khong co file nao duoc xuat.")


=== Ket qua xuat file ===
  Province  Records                    File
  An Giang      606   An_Giang_20260319.csv
  Bac Lieu      368   Bac_Lieu_20260319.csv
   Ben Tre      429    Ben_Tre_20260319.csv
    Ca Mau      588     Ca_Mau_20260319.csv
   Can Tho      572    Can_Tho_20260319.csv
 Hau Giang      377  Hau_Giang_20260319.csv
Kien Giang      797 Kien_Giang_20260319.csv
   Long An      145    Long_An_20260319.csv
 Soc Trang      529  Soc_Trang_20260319.csv
Tien Giang      512 Tien_Giang_20260319.csv
  Tra Vinh      141   Tra_Vinh_20260319.csv
 Vinh Long      369  Vinh_Long_20260319.csv

Tong records DBSCL: 5,433
Thu muc output    : d:\Hoc_tap\PROJECT_WEATHER_FORCAST\data\data_dbscl

Cac tinh CHUA CO du lieu (1):
  - Dong Thap
